# 0. duckdb 세팅하기

### 0. 설치하기

In [1]:
!pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.8 MB/s eta 0:00:0000:0100:01


In [16]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable


### 1. 세팅하기

In [30]:
import duckdb as dd
import dotenv
import os
import pandas as pd

In [20]:
# .env 파일 경로 찾기

env_path = dotenv.find_dotenv()

In [21]:
# .env 파일 불러오기  (내용이 있으면 -> True, 없으면 -> False)

dotenv.load_dotenv(
    dotenv_path=env_path,
    override=True
)

True

In [5]:
mem_con = dd.connect("mydb.duckdb")

In [26]:
HMAC_ID = os.getenv("HMAC_ID")
HMAC_PW = os.getenv("HMAC_PW")

secret_gcs = f"""
CREATE SECRET (
    TYPE GCS,
    KEY_ID '{HMAC_ID}',
    SECRET '{HMAC_PW}'
);
"""

mem_con.execute(secret_gcs)

In [27]:
mem_con.execute("FROM duckdb_secrets()").df()

,name,type,provider,persistent,storage,scope,secret_string
0,__default_gcs,gcs,config,False,memory,"[gcs://, gs://]",name=__default_gcs;type=gcs;provider=config;se...


### 2. 파일 불러오기

In [31]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/accounts_attendance.parquet')
"""

attendance_df = mem_con.execute(_query).df()

In [32]:
attendance_df.head()

,id,attendance_date_list,user_id
0,1,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",1446852
1,2,"[""2023-05-27"", ""2023-05-29"", ""2023-05-30"", ""20...",1359398
2,3,"[""2023-05-27"", ""2023-05-29"", ""2023-05-30"", ""20...",1501542
3,4,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",1507767
4,5,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",1287453


In [33]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/polls_questionpiece.parquet')
"""

polls_questionpiece_df = mem_con.execute(_query).df()

In [34]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/polls_questionset.parquet')
"""

polls_questionset_df = mem_con.execute(_query).df()

In [53]:
_query = """
FROM read_parquet('gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/polls_question.parquet')
"""

polls_question_df = mem_con.execute(_query).df()

# 2. 피쳐엔지니어링

### 2) 질문테이블 : 질문piece, 질문set, 질문 테이블 병합

questionset 1정규화

In [36]:
polls_questionset_df.head()

,id,question_piece_id_list,opening_time,status,created_at,user_id
0,99817,"[998458, 998459, 998460, 998461, 998462, 99846...",2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
1,99830,"[998588, 998589, 998590, 998591, 998592, 99859...",2023-04-28 12:28:07,F,2023-04-28 12:28:07,849438
2,99840,"[998689, 998691, 998693, 998695, 998697, 99869...",2023-04-28 12:28:38,F,2023-04-28 12:28:38,847375
3,99841,"[998688, 998690, 998692, 998694, 998696, 99869...",2023-04-28 12:28:38,F,2023-04-28 12:28:38,849446
4,99848,"[998768, 998769, 998770, 998771, 998772, 99877...",2023-04-28 12:28:57,F,2023-04-28 12:28:57,849477


In [48]:
print("shape : ", polls_questionpiece_df.shape)
print("id 갯수 : ", polls_questionpiece_df['id'].nunique())

shape :  (1265476, 5)
id 갯수 :  1265476


In [37]:
# 정규화 대상 DataFrame
df = polls_questionset_df.copy()

# 리스트 컬럼 정규화 (explode 사용)
df['question_piece_id_list'] = df['question_piece_id_list'].apply(eval)  # 문자열로 저장된 리스트를 진짜 리스트로 변환
normalized_df = df.explode('question_piece_id_list')[['id', 'question_piece_id_list']]

# 열 이름 바꾸기 (선택사항)
normalized_df = normalized_df.rename(columns={'question_piece_id_list': 'question_piece_id'})

# 결과 확인
normalized_df.head()

,id,question_piece_id
0,99817,998458
0,99817,998459
0,99817,998460
0,99817,998461
0,99817,998462


In [40]:
merged_df = normalized_df.merge(
    df[['id', 'opening_time', 'status', 'created_at', 'user_id']],
    how='left',
    on='id'
)

In [58]:
# 정규화 완료 테이블
merged_df.head()

,id,question_piece_id,opening_time,status,created_at,user_id
0,99817,998458,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
1,99817,998459,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
2,99817,998460,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
3,99817,998461,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
4,99817,998462,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436


In [45]:
# question_piece_id 중복값 확인
merged_df['question_piece_id'].duplicated().sum()

0

In [63]:
merged_df['question_piece_id'].nunique()

1583840

questionpiece 테이블 합치기

In [35]:
polls_questionpiece_df.head()

,id,is_voted,created_at,question_id,is_skipped
0,998458,1,2023-04-28 12:27:22,252,0
1,998459,1,2023-04-28 12:27:22,244,0
2,998460,1,2023-04-28 12:27:22,183,0
3,998461,1,2023-04-28 12:27:22,101,0
4,998462,1,2023-04-28 12:27:22,209,0


In [41]:
merged_df.head()

,id,question_piece_id,opening_time,status,created_at,user_id
0,99817,998458,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
1,99817,998459,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
2,99817,998460,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
3,99817,998461,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
4,99817,998462,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436


In [51]:
question_df = merged_df.merge(
    polls_questionpiece_df, 
    how='left', 
    left_on='question_piece_id', 
    right_on='id',
    suffixes=('_set','_piece'))

In [60]:
question_df.head(20)

,id_set,question_piece_id,opening_time,status,created_at_set,user_id,id_piece,is_voted,created_at_piece,question_id,is_skipped
0,99817,998458,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998458.0,1.0,2023-04-28 12:27:22,252.0,0.0
1,99817,998459,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998459.0,1.0,2023-04-28 12:27:22,244.0,0.0
2,99817,998460,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998460.0,1.0,2023-04-28 12:27:22,183.0,0.0
3,99817,998461,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998461.0,1.0,2023-04-28 12:27:22,101.0,0.0
4,99817,998462,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998462.0,1.0,2023-04-28 12:27:22,209.0,0.0
5,99817,998463,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998463.0,1.0,2023-04-28 12:27:22,239.0,0.0
6,99817,998464,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998464.0,1.0,2023-04-28 12:27:22,146.0,0.0
7,99817,998465,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998465.0,1.0,2023-04-28 12:27:22,297.0,0.0
8,99817,998466,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998466.0,1.0,2023-04-28 12:27:22,294.0,0.0
9,99817,998467,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998467.0,1.0,2023-04-28 12:27:22,201.0,0.0


In [62]:
question_df.isna().sum()

id_set                    0
question_piece_id         0
opening_time              0
status                    0
created_at_set            0
user_id                   0
id_piece             318364
is_voted             318364
created_at_piece     318364
question_id          318364
is_skipped           318364
dtype: int64

In [65]:
nan_rows = question_df[question_df.isna().any(axis=1)]
nan_rows

,id_set,question_piece_id,opening_time,status,created_at_set,user_id,id_piece,is_voted,created_at_piece,question_id,is_skipped
18,99830,998596,2023-04-28 12:28:07,F,2023-04-28 12:28:07,849438,NaN,NaN,NaT,NaN,NaN
30,99841,998688,2023-04-28 12:28:38,F,2023-04-28 12:28:38,849446,NaN,NaN,NaT,NaN,NaN
31,99841,998690,2023-04-28 12:28:38,F,2023-04-28 12:28:38,849446,NaN,NaN,NaT,NaN,NaN
32,99841,998692,2023-04-28 12:28:38,F,2023-04-28 12:28:38,849446,NaN,NaN,NaT,NaN,NaN
35,99841,998698,2023-04-28 12:28:38,F,2023-04-28 12:28:38,849446,NaN,NaN,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1583694,20835070,208351465,2024-03-19 12:53:58,F,2024-03-19 12:53:58,1583358,NaN,NaN,NaT,NaN,NaN
1583695,20835070,208351466,2024-03-19 12:53:58,F,2024-03-19 12:53:58,1583358,NaN,NaN,NaT,NaN,NaN
1583696,20835070,208351467,2024-03-19 12:53:58,F,2024-03-19 12:53:58,1583358,NaN,NaN,NaT,NaN,NaN
1583698,20835070,208351469,2024-03-19 12:53:58,F,2024-03-19 12:53:58,1583358,NaN,NaN,NaT,NaN,NaN


In [67]:
nan_rows['status'].value_counts()

status
F    318351
O        11
C         2
Name: count, dtype: int64

polls_question_df랑 merge

In [54]:
polls_question_df.head()

,id,question_text,created_at
0,99,가장 신비한 매력이 있는 사람은?,2023-03-31 15:22:53
1,100,"""이 사람으로 한 번 살아보고 싶다"" 하는 사람은?",2023-03-31 15:22:53
2,101,미래의 틱톡커는?,2023-03-31 15:22:54
3,102,여기서 제일 특이한 친구는?,2023-03-31 15:22:54
4,103,가장 지켜주고 싶은 사람은?,2023-03-31 15:22:55


In [55]:
question_df2 = question_df.merge(
  polls_question_df,
  how='left',
  left_on='question_id',
  right_on='id'
)

In [57]:
question_df2.head()

,id_set,question_piece_id,opening_time,status,created_at_set,user_id,id_piece,is_voted,created_at_piece,question_id,is_skipped,id,question_text,created_at
0,99817,998458,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998458.0,1.0,2023-04-28 12:27:22,252.0,0.0,252.0,손이 가장 이쁘게 생겼을거 같은 사람은?,2023-04-01 11:09:27
1,99817,998459,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998459.0,1.0,2023-04-28 12:27:22,244.0,0.0,244.0,대학교에서 학생회장할 것 같은 사람은?,2023-04-01 11:09:26
2,99817,998460,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998460.0,1.0,2023-04-28 12:27:22,183.0,0.0,183.0,나의 자존감을 가장 많이 높여줬던 사람은?,2023-04-01 11:09:14
3,99817,998461,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998461.0,1.0,2023-04-28 12:27:22,101.0,0.0,101.0,미래의 틱톡커는?,2023-03-31 15:22:54
4,99817,998462,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998462.0,1.0,2023-04-28 12:27:22,209.0,0.0,209.0,항상 좋은 냄새가 나는 사람은?,2023-04-01 11:09:20


null 제거

In [66]:
question_df2.isna().sum()

id_set                    0
question_piece_id         0
opening_time              0
status                    0
created_at_set            0
user_id                   0
id_piece             318364
is_voted             318364
created_at_piece     318364
question_id          318364
is_skipped           318364
id                   318364
question_text        318364
created_at           318364
dtype: int64

In [68]:
question_df_notnull = question_df2.dropna()
question_df_notnull

,id_set,question_piece_id,opening_time,status,created_at_set,user_id,id_piece,is_voted,created_at_piece,question_id,is_skipped,id,question_text,created_at
0,99817,998458,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998458.0,1.0,2023-04-28 12:27:22,252.0,0.0,252.0,손이 가장 이쁘게 생겼을거 같은 사람은?,2023-04-01 11:09:27
1,99817,998459,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998459.0,1.0,2023-04-28 12:27:22,244.0,0.0,244.0,대학교에서 학생회장할 것 같은 사람은?,2023-04-01 11:09:26
2,99817,998460,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998460.0,1.0,2023-04-28 12:27:22,183.0,0.0,183.0,나의 자존감을 가장 많이 높여줬던 사람은?,2023-04-01 11:09:14
3,99817,998461,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998461.0,1.0,2023-04-28 12:27:22,101.0,0.0,101.0,미래의 틱톡커는?,2023-03-31 15:22:54
4,99817,998462,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,998462.0,1.0,2023-04-28 12:27:22,209.0,0.0,209.0,항상 좋은 냄새가 나는 사람은?,2023-04-01 11:09:20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1583835,20838446,208385226,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,208385226.0,0.0,2024-05-07 11:32:30,960.0,0.0,960.0,가장 인싸일 것 같은 친구는?,2023-05-15 14:00:19
1583836,20838446,208385227,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,208385227.0,0.0,2024-05-07 11:32:30,1402.0,0.0,1402.0,가장 리더쉽 있을 것 같은 사람,2023-05-15 14:03:19
1583837,20838446,208385228,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,208385228.0,0.0,2024-05-07 11:32:30,1676.0,0.0,1676.0,화장 전후 가장 비슷할 것 같은 사람은?,2023-06-02 08:06:24
1583838,20838446,208385229,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,208385229.0,0.0,2024-05-07 11:32:30,3115.0,0.0,3115.0,가장 박새로이컷이 잘 어울릴 것 같은 사람,2023-06-02 08:07:08


In [69]:
question_df_notnull.shape

(1265476, 14)

columns 정리

In [70]:
question_df_notnull.columns

Index(['id_set', 'question_piece_id', 'opening_time', 'status',
       'created_at_set', 'user_id', 'id_piece', 'is_voted', 'created_at_piece',
       'question_id', 'is_skipped', 'id', 'question_text', 'created_at'],
      dtype='object')

In [73]:
final_question_df = question_df_notnull[['id_set', 'question_piece_id', 'opening_time', 'status',
                                          'created_at_set', 'user_id', 'is_voted', 'is_skipped', 'created_at_piece',
                                          'question_id',  'question_text']]

In [74]:
final_question_df

,id_set,question_piece_id,opening_time,status,created_at_set,user_id,is_voted,is_skipped,created_at_piece,question_id,question_text
0,99817,998458,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,1.0,0.0,2023-04-28 12:27:22,252.0,손이 가장 이쁘게 생겼을거 같은 사람은?
1,99817,998459,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,1.0,0.0,2023-04-28 12:27:22,244.0,대학교에서 학생회장할 것 같은 사람은?
2,99817,998460,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,1.0,0.0,2023-04-28 12:27:22,183.0,나의 자존감을 가장 많이 높여줬던 사람은?
3,99817,998461,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,1.0,0.0,2023-04-28 12:27:22,101.0,미래의 틱톡커는?
4,99817,998462,2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436,1.0,0.0,2023-04-28 12:27:22,209.0,항상 좋은 냄새가 나는 사람은?
...,...,...,...,...,...,...,...,...,...,...,...
1583835,20838446,208385226,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,0.0,0.0,2024-05-07 11:32:30,960.0,가장 인싸일 것 같은 친구는?
1583836,20838446,208385227,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,0.0,0.0,2024-05-07 11:32:30,1402.0,가장 리더쉽 있을 것 같은 사람
1583837,20838446,208385228,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,0.0,0.0,2024-05-07 11:32:30,1676.0,화장 전후 가장 비슷할 것 같은 사람은?
1583838,20838446,208385229,2024-05-07 12:12:30,C,2024-05-07 11:32:30,945560,0.0,0.0,2024-05-07 11:32:30,3115.0,가장 박새로이컷이 잘 어울릴 것 같은 사람


추출하기

In [75]:
mem_con.execute("""
                COPY final_question_df TO 'gs://sprintda07-gohee-bucket/final_project/최종프로젝트_데이터(parquet)/votes/question_df.parquet' (FORMAT PARQUET);
                """)